![Banner](https://i.imgur.com/a3uAqnb.png)

# Double DQN Implementation - Homework Assignment

In this homework, you will implement **Double Deep Q-Learning** using PyTorch to train an agent to play the Lunar Lander game. This will involve understanding reinforcement learning concepts, implementing experience replay, and optimizing the training process.

## 📌 Project Overview
- **Task**: Train an RL agent to land a spacecraft safely
- **Algorithm**: Double Deep Q-Network (DDQN)
- **Environment**: LunarLander-v2 from OpenAI Gym
- **Goal**: Achieve consistent successful landings (score > 200)

## 📚 Learning Objectives
By completing this assignment, you will:
- Understand Q-Learning and temporal difference methods
- Implement experience replay for stable training
- Use target networks to reduce correlation
- Apply Double DQN to reduce maximization bias
- Evaluate reinforcement learning agents
- Visualize training progress and agent behavior# Double DQN

## Lunar Lander

This environment is a classic rocket trajectory optimization problem. The landing pad is always at coordinates (0,0). The state is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

There are four discrete actions available:<br>
- 0: do nothing<br>
- 1: fire left orientation engine<br>
- 2: fire main engine<br>
- 3: fire right orientation engine<br>

After every step a reward is granted. The total reward of an episode is the sum of the rewards for all the steps within that episode.

For each step, the reward:

- is increased/decreased the closer/further the lander is to the landing pad.

- is increased/decreased the slower/faster the lander is moving.

- is decreased the more the lander is tilted (angle not horizontal).

- is increased by 10 points for each leg that is in contact with the ground.

- is decreased by 0.03 points each frame a side engine is firing.

- is decreased by 0.3 points each frame the main engine is firing.

The episode receive an additional reward of -100 or +100 points for crashing or landing safely respectively.

An episode is considered a solution if it scores at least 200 points.


You can read more the LunarLander environment [here](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

![LunarLander](https://gymnasium.farama.org/_images/lunar_lander.gif)

## Deep Q-Learning

The main idea behind Q-learning is that if we had a function
$Q^*: State \times Action \rightarrow \mathbb{R}$, that could tell
us what our return would be, if we were to take an action in a given
state, then we could easily construct a policy that maximizes our
rewards:

\begin{align}\pi^*(s) = \arg\!\max_a \ Q^*(s, a)\end{align}

But this is not scalable. Must compute $Q(s,a)$ for every state-action pair. If state is e.g. current game state pixels, computationally infeasible to compute for entire state space! But, since neural networks are universal function
approximators, we can simply create one and train it to resemble
$Q^*$.

For our training update rule, we'll use a fact that every $Q$
function for some policy obeys the Bellman equation:

\begin{align}Q^{\pi}(s, a) = r + \gamma Q^{\pi}(s', \pi(s'))\end{align}

The difference between the two sides of the equality is known as the
temporal difference error, $\delta$:

\begin{align}\delta = Q(s, a) - (r + \gamma \max_a Q(s', a))\end{align}

To minimise this error, we will use the `Huber
loss <https://en.wikipedia.org/wiki/Huber_loss>`__. The Huber loss acts
like the mean squared error when the error is small, but like the mean
absolute error when the error is large - this makes it more robust to
outliers when the estimates of $Q$ are very noisy. We calculate
this over a batch of transitions, $B$, sampled from the replay
memory:

\begin{align}\mathcal{L} = \frac{1}{|B|}\sum_{(s, a, s', r) \ \in \ B} \mathcal{L}(\delta)\end{align}

\begin{align}\text{where} \quad \mathcal{L}(\delta) = \begin{cases}
     \frac{1}{2}{\delta^2}  & \text{for } |\delta| \le 1, \\
     |\delta| - \frac{1}{2} & \text{otherwise.}
   \end{cases}\end{align}



### Double Deep Q-Learning

We will implement Double Deep Q-Learning here. Double Deep Q-Learning is used to reduce the maximaztion bias in Q-Learning. This entails separate action selection and action evaluation in the target value.

- Use the current network to select the max action for the next state
and then use the target network to get the target Q-value for that
action.




![ddqn.png](https://i.imgur.com/JUVRwEP.png)

[Image Source](https://leejungi.github.io/posts/Dueling-DQN/)

In [ ]:
# TODO: Install required packages (uncomment if needed):
# !pip install -q swig
# !pip install -q gym[box2d]
# !pip install -q pygame
# !pip install -q moviepy

# TODO: Import all necessary libraries:
#       - gym for the environment
#       - torch, torch.nn, torch.optim for neural networks
#       - math, random, numpy for utilities
#       - matplotlib for plotting
#       - collections.namedtuple for experience storage
#       - itertools.count for infinite counting
#       - PIL.Image for image processing
#       - torchvision.transforms for image transforms

import gym
import math
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple

from itertools import count
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T

# TODO: Set random seeds for reproducibility
# TODO: Check if GPU is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1️⃣ Environment Setup

**Task**: Create and explore the Lunar Lander environment.

**Requirements**:
- Initialize the LunarLander-v2 environment
- Understand the state space (8-dimensional vector)
- Understand the action space (4 discrete actions)
- Explore the reward structure

In [ ]:
# TODO: Create the Lunar Lander environment
#       - Use gym.make("LunarLander-v2")
#       - Print environment information
env = gym.make("LunarLander-v2")

# TODO: Print environment details:
#       - Action space size
#       - Observation space shape
#       - Reset environment and print initial state
print(f"Action space: {env.action_space}")
print(f"Action space size: {env.action_space.n}")
print(f"Observation space: {env.observation_space}")
print(f"Observation space shape: {env.observation_space.shape}")

# TODO: Reset environment and examine initial state
state, info = env.reset()
print(f"Initial state shape: {state.shape}")
print(f"Sample state: {state}")

# TODO: Explore one random step
action = env.action_space.sample()
next_state, reward, terminated, truncated, info = env.step(action)
print(f"After random action {action}:")
print(f"Next state: {next_state}")
print(f"Reward: {reward}")
print(f"Terminated: {terminated}")

## 2️⃣ Experience Replay Memory

**Task**: Implement experience replay to store and sample transitions.

**Requirements**:
- Create a namedtuple for storing transitions (state, action, next_state, reward)
- Implement a circular buffer with fixed capacity
- Add methods for storing and sampling experiences
- Ensure random sampling to break correlation between consecutive experiences

Learning from batches of consecutive samples is problematic as the sample are correlated and it can create a bad feedback loop if one action is dominated in the samples.

We can address these problems using an experience replay memory. It maintains a record for all the transitions experienced. The agent is then trained by sampling random minibatches from the replay memory.

In [ ]:
# TODO: Define Transition namedtuple with fields:
#       - state: current state
#       - action: action taken
#       - next_state: resulting state (None if terminal)
#       - reward: immediate reward received
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))

# TODO: Implement ReplayMemory class:
class ReplayMemory(object):
    def __init__(self, capacity):
        # TODO: Initialize memory buffer and position counter
        #       - self.capacity: maximum number of experiences to store
        #       - self.memory: list to store transitions
        #       - self.position: current position in circular buffer
        self.capacity = capacity
        self.memory = []
        self.position = 0

    def push(self, *args):
        """Saves a transition."""
        # TODO: Add transition to memory:
        #       - If memory not full, append None
        #       - Store transition at current position
        #       - Update position with wrap-around
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        self.memory[self.position] = Transition(*args)
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        # TODO: Return random sample of transitions
        #       - Use random.sample() to get batch_size transitions
        return random.sample(self.memory, batch_size)

    def __len__(self):
        # TODO: Return current size of memory
        return len(self.memory)

# TODO: Test the replay memory implementation
print("Testing ReplayMemory...")
memory = ReplayMemory(1000)
print(f"Initial memory size: {len(memory)}")

# Add some dummy transitions
for i in range(5):
    state = torch.randn(8)
    action = torch.tensor([[i % 4]])
    next_state = torch.randn(8) if i < 4 else None
    reward = torch.tensor([float(i)])
    memory.push(state, action, next_state, reward)

print(f"Memory size after 5 pushes: {len(memory)}")

## 3️⃣ Deep Q-Network Architecture

**Task**: Design the neural network that will approximate the Q-function.

**Requirements**:
- Create a feedforward neural network with appropriate layers
- Input: 8-dimensional state vector (LunarLander observations)
- Output: 4-dimensional vector (Q-values for each action)
- Use ReLU activations and appropriate layer sizes

In [ ]:
# TODO: Implement DQN class inheriting from nn.Module:
class DQN(nn.Module):
    def __init__(self, n_observations, n_actions):
        # TODO: Initialize the network layers:
        #       - layer1: Linear(n_observations, 128) 
        #       - layer2: Linear(128, 128)
        #       - layer3: Linear(128, n_actions)
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    def forward(self, x):
        # TODO: Implement forward pass:
        #       - Apply ReLU activation to first two layers
        #       - No activation on final layer (raw Q-values)
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

# TODO: Test the network architecture
n_actions = env.action_space.n
state, info = env.reset()
n_observations = len(state)

print(f"Network input size: {n_observations}")
print(f"Network output size: {n_actions}")

# Create test network
test_net = DQN(n_observations, n_actions).to(device)
test_input = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
test_output = test_net(test_input)
print(f"Test input shape: {test_input.shape}")
print(f"Test output shape: {test_output.shape}")
print(f"Sample Q-values: {test_output}")

## 4️⃣ Exploration Strategy

**Task**: Implement epsilon-greedy exploration strategy.

**Requirements**:
- Start with high exploration (epsilon = 0.9)
- Gradually decay to low exploration (epsilon = 0.05)
- Use exponential decay schedule
- Balance exploration vs exploitation during training

In [ ]:
# TODO: Implement epsilon-greedy action selection function:
def select_action(state, policy_net, steps_done, eps_start=0.9, eps_end=0.05, eps_decay=1000):
    """
    Select action using epsilon-greedy policy.
    
    Args:
        state: Current state tensor
        policy_net: Policy network for Q-value prediction
        steps_done: Number of steps taken so far
        eps_start: Initial epsilon value
        eps_end: Final epsilon value  
        eps_decay: Decay rate for epsilon
    
    Returns:
        Action tensor (1x1)
    """
    # TODO: Calculate current epsilon using exponential decay:
    #       eps_threshold = eps_end + (eps_start - eps_end) * exp(-steps_done / eps_decay)
    sample = random.random()
    eps_threshold = eps_end + (eps_start - eps_end) * \
        math.exp(-1. * steps_done / eps_decay)
    
    # TODO: Choose action based on epsilon-greedy policy:
    #       - If random sample > epsilon: choose greedy action (exploitation)
    #       - Else: choose random action (exploration)
    if sample > eps_threshold:
        with torch.no_grad():
            # TODO: Get Q-values from policy network
            #       Return action with highest Q-value
            return policy_net(state).max(1)[1].view(1, 1)
    else:
        # TODO: Return random action
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

# TODO: Test exploration strategy
print("Testing exploration strategy...")
test_net = DQN(n_observations, n_actions).to(device)
test_state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

# Test at different steps to see epsilon decay
for steps in [0, 100, 500, 1000, 2000]:
    action = select_action(test_state, test_net, steps)
    eps = 0.05 + (0.9 - 0.05) * math.exp(-1. * steps / 1000)
    print(f"Steps: {steps}, Epsilon: {eps:.3f}, Action: {action.item()}")

## 5️⃣ Hyperparameters and Network Initialization

**Task**: Set up hyperparameters and initialize the networks.

**Requirements**:
- Define appropriate hyperparameters for stable training
- Initialize policy network and target network
- Set up optimizer with suitable learning rate
- Initialize replay memory with sufficient capacity

In [ ]:
# TODO: Define hyperparameters:
# BATCH_SIZE: Number of transitions sampled from replay buffer
# GAMMA: Discount factor for future rewards (0.99 is standard)
# EPS_START/EPS_END/EPS_DECAY: Epsilon-greedy parameters
# TAU: Soft update rate for target network (0.005 for slow updates)
# LR: Learning rate for optimizer
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 1000
TAU = 0.005
LR = 1e-4

# TODO: Get environment dimensions
n_actions = env.action_space.n
state, info = env.reset()
n_observations = len(state)

# TODO: Initialize networks:
#       - policy_net: Main network for action selection
#       - target_net: Stable target for computing target Q-values
#       - Copy policy_net weights to target_net initially
policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

# TODO: Initialize optimizer and replay memory:
#       - Use AdamW optimizer with amsgrad=True
#       - Create replay memory with capacity 10000
optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)

# TODO: Initialize step counter
steps_done = 0

## 6️⃣ Double DQN Training Function

**Task**: Implement the core training function with Double DQN updates.

**Requirements**:
- Sample batch from replay memory
- Compute current Q-values using policy network
- Compute target Q-values using Double DQN approach
- Use policy network for action selection, target network for Q-value estimation
- Minimize Huber loss between current and target Q-values

In [ ]:
def optimize_model():
    """
    Perform one step of optimization on the policy network.
    Implements Double DQN algorithm.
    """
    # TODO: Check if we have enough samples in memory
    if len(memory) < BATCH_SIZE:
        return
    
    # TODO: Sample random batch from replay memory
    transitions = memory.sample(BATCH_SIZE)
    batch = Transition(*zip(*transitions))

    # TODO: Create mask for non-final states (states that have next_state)
    #       non_final_mask: Boolean tensor indicating which states are non-terminal
    #       non_final_next_states: Tensor of non-terminal next states
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                          batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state
                                                if s is not None])
    
    # TODO: Concatenate batch elements into tensors
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # TODO: Compute current Q-values Q(s,a):
    #       - Pass state_batch through policy_net
    #       - Use gather() to select Q-values for taken actions
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # TODO: Compute target Q-values using Double DQN:
    #       1. Use policy network to select best actions for next states
    #       2. Use target network to evaluate Q-values for selected actions
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        # Double DQN: Use policy network to select actions
        next_state_actions = policy_net(non_final_next_states).max(1)[1].detach().unsqueeze(1)
        # Use target network to evaluate the Q-values
        next_state_values[non_final_mask] = target_net(non_final_next_states).gather(1, next_state_actions).squeeze()

    # TODO: Compute target Q-values: reward + γ * max_a Q_target(s', a)
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # TODO: Compute loss and perform optimization:
    #       - Use SmoothL1Loss (Huber loss) for stability
    #       - Backpropagate and clip gradients
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

# TODO: Test the optimization function
print("Testing optimize_model function...")

# Add some dummy experiences to memory
for i in range(BATCH_SIZE + 10):
    state = torch.randn(1, n_observations, device=device)
    action = torch.tensor([[i % n_actions]], device=device)
    next_state = torch.randn(1, n_observations, device=device) if i < BATCH_SIZE else None
    reward = torch.tensor([random.random() - 0.5], device=device)
    memory.push(state, action, next_state, reward)

print(f"Memory size: {len(memory)}")
print("Running optimization step...")
optimize_model()
print("Optimization completed successfully!")

## 7️⃣ Training Loop

**Task**: Implement the main training loop for the DQN agent.

**Requirements**:
- Run training for sufficient episodes (600 episodes)
- Collect experiences and store in replay memory
- Perform optimization steps regularly
- Update target network using soft updates
- Track episode durations and rewards
- Print progress every 50 episodes

In [ ]:
# TODO: Set training parameters
num_episodes = 600

# TODO: Initialize tracking lists
episode_durations = []
episode_rewards = []

print("Starting DQN Training...")
print("=" * 50)

# TODO: Main training loop
for i_episode in range(num_episodes):
    # TODO: Reset environment and initialize state
    state, info = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    total_reward = 0
    
    # TODO: Episode loop - continue until done
    for t in count():
        # TODO: Select action using epsilon-greedy policy
        action = select_action(state, policy_net, steps_done)
        steps_done += 1
        
        # TODO: Execute action in environment
        observation, reward, terminated, truncated, _ = env.step(action.item())
        total_reward += reward
        reward = torch.tensor([reward], device=device)
        done = terminated or truncated

        # TODO: Process next state
        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # TODO: Store transition in replay memory
        memory.push(state, action, next_state, reward)

        # TODO: Update state
        state = next_state

        # TODO: Perform optimization step
        optimize_model()

        # TODO: Soft update target network:
        #       θ_target = τ * θ_policy + (1 - τ) * θ_target
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
        target_net.load_state_dict(target_net_state_dict)

        # TODO: Check if episode is done
        if done:
            episode_durations.append(t + 1)
            episode_rewards.append(total_reward)
            
            # TODO: Print progress every 50 episodes
            if i_episode % 50 == 0:
                avg_reward = np.mean(episode_rewards[-50:]) if len(episode_rewards) >= 50 else np.mean(episode_rewards)
                current_epsilon = EPS_END + (EPS_START - EPS_END) * math.exp(-1. * steps_done / EPS_DECAY)
                print(f'Episode {i_episode:3d} | Duration: {t + 1:3d} | '
                      f'Reward: {total_reward:7.2f} | Avg Reward: {avg_reward:7.2f} | '
                      f'Epsilon: {current_epsilon:.3f}')
            break

print("\nTraining Complete!")

## 8️⃣ Training Visualization

**Task**: Create comprehensive visualizations of the training progress.

**Requirements**:
- Plot episode durations over time
- Plot episode rewards over time
- Show epsilone decay over time(Steps not episodes)

In [ ]:
# TODO: Create comprehensive training visualization
plt.figure(figsize=(15, 10))

# TODO: Plot episode durations
plt.subplot(2, 2, 1)
plt.title('Episode Durations', fontsize=14, fontweight='bold')
plt.xlabel('Episode')
plt.ylabel('Duration (steps)')
plt.plot(episode_durations, alpha=0.6, color='blue', linewidth=0.8)


plt.grid(True, alpha=0.3)

# TODO: Plot episode rewards
plt.subplot(2, 2, 2)
plt.title('Episode Rewards', fontsize=14, fontweight='bold')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.plot(episode_rewards, alpha=0.6, color='green', linewidth=0.8)


# TODO: Plot epsilon decay
plt.subplot(2, 2, 4)
plt.title('Epsilon Decay', fontsize=14, fontweight='bold')
plt.xlabel('Steps')
plt.ylabel('Epsilon Value')

steps_range = range(0, steps_done, steps_done//100)
epsilon_values = [EPS_END + (EPS_START - EPS_END) * math.exp(-1. * s / EPS_DECAY) 
                  for s in steps_range]
plt.plot(steps_range, epsilon_values, color='orange', linewidth=2)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9️⃣ Agent Testing and Visualization

**Task**: Test the trained agent and create a video visualization.

**Requirements**:
- Test the trained agent without exploration (greedy policy)
- Record the agent's performance in the environment
- Create a video of the agent playing

In [ ]:
# TODO: Set up video recording environment
from gym.wrappers.monitoring import video_recorder
from IPython.display import HTML
from IPython import display
import glob
import base64, io, os

# Set up environment for video recording
os.environ['SDL_VIDEODRIVER']='dummy'

def show_video():
    """Display recorded video in Jupyter notebook."""
    mp4list = glob.glob('video*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display.display(HTML(data='''<video alt="Trained DQN Agent" autoplay 
                loop controls style=\"height: 400px;\">
                <source src=\"data:video/mp4;base64,{0}\" type=\"video/mp4\" />
             </video>'''.format(encoded.decode('ascii'))))
    else: 
        print("Could not find video")

def wrap_env(env):
    """Wrap environment for video recording."""
    env = video_recorder.VideoRecorder(env, "video.mp4")
    return env

# TODO: Test the trained agent
print("Testing Trained DQN Agent...")
print("="*40)

# Create test environment with rendering
env_test = gym.make("LunarLander-v2", render_mode="rgb_array")
vid = wrap_env(env_test)

# TODO: Run test episode
state, info = env_test.reset()
state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
total_reward = 0
episode_steps = 0

for t in range(1000):  # Maximum steps per episode
    # TODO: Record frame for video
    vid.capture_frame()
    
    # TODO: Use trained policy network (no exploration - greedy policy)
    with torch.no_grad():
        q_values = policy_net(state)
        action = q_values.max(1)[1].view(1, 1)
    
    # TODO: Execute action
    observation, reward, terminated, truncated, _ = env_test.step(action.item())
    total_reward += reward
    episode_steps += 1
    
    # TODO: Check if episode finished
    if terminated or truncated:
        print(f"Episode finished after {episode_steps} timesteps")
        print(f"Total reward: {total_reward:.2f}")
        
        # Analyze performance
        if total_reward >= 200:
            print("🎉 SUCCESS: Agent achieved landing score ≥ 200!")
        elif total_reward >= 0:
            print("👍 GOOD: Agent achieved positive score")
        else:
            print("❌ POOR: Agent crashed or performed poorly")
        break   
    
    # TODO: Update state
    state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

# TODO: Clean up
vid.close()
env_test.close()

In [ ]:
# TODO: Display the recorded video
show_video()

## 1️⃣0️⃣ Action Selection Pattern Analysis

**Task**: Analyze the agent's action selection patterns and preferences.

**Requirements**:
- Track action frequencies during test episodes
- Visualize action distribution patterns

In [ ]:
# TODO: Analyze action selection patterns
print("Analyzing Action Selection Patterns...")
print("="*45)

action_names = ['Do Nothing', 'Fire Left', 'Fire Main', 'Fire Right']
action_counts = {name: 0 for name in action_names}
state_action_history = []

# TODO: Run episode and track all actions
env_test = gym.make("LunarLander-v2")
state, info = env_test.reset()
state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

for t in range(1000):
    # TODO: Get Q-values and select action
    with torch.no_grad():
        q_values = policy_net(state_tensor)
        action = q_values.max(1)[1].item()
    
    # TODO: Track action and state
    action_counts[action_names[action]] += 1
    state_action_history.append({
        'step': t,
        'state': state_tensor.cpu().numpy().flatten(),
        'action': action,
        'q_values': q_values.cpu().numpy().flatten()
    })
    
    # TODO: Execute action
    observation, reward, terminated, truncated, _ = env_test.step(action)
    
    if terminated or truncated:
        break
        
    state_tensor = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

env_test.close()

# TODO: Visualize action distribution
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.title('Action Selection Frequency', fontsize=14, fontweight='bold')
actions = list(action_counts.keys())
counts = list(action_counts.values())
colors = ['red', 'blue', 'orange', 'green']
bars = plt.bar(actions, counts, color=colors, alpha=0.7)
plt.ylabel('Frequency')
plt.xticks(rotation=45)

# Add value labels on bars
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', va='bottom', fontweight='bold')

plt.grid(True, alpha=0.3)

# TODO: Plot action selection over time
plt.subplot(2, 2, 2)
plt.title('Action Selection Over Time', fontsize=14, fontweight='bold')
steps = [entry['step'] for entry in state_action_history]
actions_taken = [entry['action'] for entry in state_action_history]
plt.scatter(steps, actions_taken, alpha=0.6, c=actions_taken, cmap='viridis')
plt.ylabel('Action')
plt.xlabel('Time Step')
plt.yticks([0, 1, 2, 3], action_names)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 📋 Assignment Evaluation Criteria

Your DQN homework will be evaluated based on the following criteria:

### **Implementation Correctness (40%)**
- ✅ Proper Double DQN algorithm implementation
- ✅ Correct experience replay mechanism
- ✅ Appropriate network architecture and training loop
- ✅ Proper epsilon-greedy exploration strategy
- ✅ Target network updates and soft copying

### **Training Performance (25%)**
- ✅ Agent trains without errors for specified episodes
- ✅ Achieves reasonable performance (average reward > 100) - After Training
- ✅ Shows clear learning progress over time
- ✅ Proper use of hyperparameters

### **Code Quality and Documentation (20%)**
- ✅ Clean, readable code with comprehensive comments
- ✅ Proper tensor handling and device management
- ✅ Efficient implementation without memory leaks
- ✅ Well-structured functions and classes

### **Analysis and Understanding (15%)**
- ✅ Comprehensive training visualizations
